<a href="https://colab.research.google.com/github/MDShaddoud/lumiere-rag/blob/main/lumiere_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q google-genai numpy

from google import genai
from google.genai import types
from google.colab import userdata
import numpy as np

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print("Connected ✅")

Connected ✅


In [4]:
models = [m.name for m in client.models.list()]
embed_models = [m for m in models if "embedding" in m]
gen_models = [m for m in models if "flash" in m and not any(x in m for x in ["embedding","image","tts","live","audio"])]

print("Embedding models:", embed_models)
print("Chat models:", gen_models)

EMBED_MODEL = embed_models[0]
GEN_MODEL = gen_models[0]
print("\nUsing:", EMBED_MODEL, "|", GEN_MODEL)

Embedding models: ['models/gemini-embedding-001', 'models/gemini-embedding-2-preview', 'models/gemini-embedding-2']
Chat models: ['models/gemini-2.5-flash', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-omni-1.2-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash']

Using: models/gemini-embedding-001 | models/gemini-2.5-flash


In [11]:
GEN_MODEL = "models/gemini-3.8-flash"
print("Now using:", GEN_MODEL)

Now using: models/gemini-3.8-flash


In [13]:
import time

FALLBACKS = ["models/gemini-3.8-flash"] + [m for m in gen_models if "lite" in m]
print("Will try, in order:", FALLBACKS)

def generate(prompt):
    for model in FALLBACKS:
        for attempt in range(3):
            try:
                r = client.models.generate_content(model=model, contents=prompt)
                return r.text, model
            except Exception as e:
                msg = str(e)
                if "503" in msg or "429" in msg:      # busy / rate-limited → wait and retry
                    time.sleep(5 * (attempt + 1))
                    continue
                break                                  # 404 retired model etc. → try next model
    return "⚠️ All models busy — try again in a minute.", None

def ask(question):
    hits, texts = search(question)
    context = "\n\n---\n\n".join(texts)
    prompt = f"""You are the assistant for Lumière Ladies Salon.
Answer ONLY using the context below. If the answer is not in the context, say you'll check with the team.
Reply in the same language as the question. Arabic = Syrian white dialect.

CONTEXT:
{context}

QUESTION: {question}"""
    answer, used = generate(prompt)
    print("🔎 Retrieved:", hits)
    print("🤖 Model:", used)
    print("💬 Answer:", answer, "\n")

Will try, in order: ['models/gemini-3.8-flash', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.5-flash-lite']


In [5]:
from google.colab import files
up = files.upload()   # choose lumiere-knowledge-base.txt
text = list(up.values())[0].decode("utf-8")
print(len(text), "characters loaded")

Saving lumiere-knowledge-base.txt to lumiere-knowledge-base.txt
7298 characters loaded


In [6]:
sections = text.split("\n## ")[1:]   # skip the title line
chunks = []
for s in sections:
    title, body = s.split("\n", 1)
    chunks.append({"title": title.strip(), "text": title.strip() + "\n" + body.strip()})

print(len(chunks), "chunks")
for c in chunks[:3]:
    print("-", c["title"])

21 chunks
- About the salon
- Opening hours
- Location and parking


In [7]:
res = client.models.embed_content(
    model=EMBED_MODEL,
    contents=[c["text"] for c in chunks],
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
E = np.array([e.values for e in res.embeddings])
E = E / np.linalg.norm(E, axis=1, keepdims=True)   # normalize
print("Matrix shape:", E.shape)

Matrix shape: (21, 3072)


In [8]:
def search(question, k=3):
    q = client.models.embed_content(
        model=EMBED_MODEL, contents=question,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values
    q = np.array(q); q = q / np.linalg.norm(q)
    scores = E @ q                          # cosine similarity with every chunk
    top = np.argsort(scores)[::-1][:k]      # best k
    return [(chunks[i]["title"], round(float(scores[i]), 3)) for i in top], [chunks[i]["text"] for i in top]

print(search("How long does keratin last?")[0])
print(search("شو بصير اذا تأخرت؟")[0])

[('Keratin Treatment', 0.711), ('Booking policy', 0.628), ('Highlights', 0.601)]
[('Late arrival', 0.774), ('Cancellation and rescheduling', 0.699), ('Loyalty programme', 0.668)]


In [15]:
import re

FALLBACKS = ["models/gemini-flash-lite-latest", "models/gemini-3.8-flash"]   # fast model first

def lang_of(q):
    return "Arabic (Syrian white dialect)" if re.search(r'[\u0600-\u06FF]', q) else "English"

def ask(question):
    hits, texts = search(question)
    context = "\n\n---\n\n".join(texts)
    lang = lang_of(question)
    prompt = f"""You are the assistant for Lumière Ladies Salon.
Answer ONLY using the context below. If the answer is not in the context, say you'll check with the team.
Reply ONLY in {lang}. The context contains both languages — use only the {lang} parts.

CONTEXT:
{context}

QUESTION: {question}"""
    answer, used = generate(prompt)
    print("🔎 Retrieved:", hits)
    print("🌐 Language:", lang)
    print("💬 Answer:", answer, "\n")

In [16]:
ask("How long does keratin last?")
ask("بقدر اعمل كيراتين وانا حامل؟")
ask("Do you have parking?")
ask("كم سعر الجل عالمانيكير؟")
ask("Do you do hair extensions?")   # NOT in the knowledge base

🔎 Retrieved: [('Keratin Treatment', 0.711), ('Booking policy', 0.628), ('Highlights', 0.601)]
🌐 Language: English
💬 Answer: Results last 3 to 5 months depending on hair care. 

🔎 Retrieved: [('Keratin Treatment', 0.725), ('Booking policy', 0.65), ('Late arrival', 0.636)]
🌐 Language: Arabic (Syrian white dialect)
💬 Answer: إذا كنتي حامل أو مرضعة، استشيري دكتورتك قبل ما تحجزي كيراتين. 

🔎 Retrieved: [('Location and parking', 0.713), ('Opening hours', 0.628), ('Payment methods', 0.62)]
🌐 Language: English
💬 Answer: Yes, free customer parking is available in front of the salon for up to 3 hours. 

🔎 Retrieved: [('Manicure', 0.764), ('Pedicure', 0.721), ('Mani + Pedi combo', 0.72)]
🌐 Language: Arabic (Syrian white dialect)
💬 Answer: سعر الجل بزيادة 31.50 درهم. 

🔎 Retrieved: [('Booking policy', 0.656), ('Children', 0.639), ('Hair Color', 0.636)]
🌐 Language: English
💬 Answer: I'll check with the team. 

